# Прокси-сервер XML-RPC

In [1]:
from collections import deque
from datetime import datetime
from time import monotonic, perf_counter, sleep
from xmlrpc.server import SimpleXMLRPCServer, SimpleXMLRPCRequestHandler
import xmlrpc.client as xmlrpc_client

MAIN_SERVER_URL = 'http://127.0.0.1:8008/RPC2'
STATS2_URL = 'http://127.0.0.1:8019/RPC2'
PROXY_HOST = '127.0.0.1'
PROXY_PORT = 8009
MAX_REQUESTS = 5
WINDOW_SECONDS = 5.0
EVENT_HISTORY_LIMIT = 1000
STATS_TIMEOUT_SECONDS = 0.5

METHODS = [
    'ping',
    'now',
    'type',
    'sum',
    'pow',
    'black_list_check_full',
    'binarize_image',
    'estimate_cloudiness',
    'send_back_binary',
    'color_inversion',
    'flip_vertical',
    'black_list_check',
]

REQUEST_TIMES = deque()
EVENTS = deque(maxlen=EVENT_HISTORY_LIMIT)

class RequestHandler(SimpleXMLRPCRequestHandler):
    rpc_paths = ('/RPC2',)

class TimeoutTransport(xmlrpc_client.Transport):
    def make_connection(self, host):
        connection = super().make_connection(host)
        connection.timeout = STATS_TIMEOUT_SECONDS
        return connection


## Rate Limiting и регистрация событий

In [2]:
def add_event(event_type, operation, event_time, wait_seconds=None,
              execution_seconds=None, status='success'):
    wait_value = None if wait_seconds is None else round(float(wait_seconds), 6)
    execution_value = None if execution_seconds is None else round(float(execution_seconds), 6)
    total_value = None
    if wait_value is not None and execution_value is not None:
        total_value = round(wait_value + execution_value, 6)

    event = {
        'event_time': event_time,
        'event_type': event_type,
        'operation': operation,
        'wait_seconds': wait_value,
        'execution_seconds': execution_value,
        'duration_seconds': total_value,
        'status': status,
    }
    EVENTS.append(event)

    try:
        with xmlrpc_client.ServerProxy(
            STATS2_URL,
            transport=TimeoutTransport(),
            allow_none=True
        ) as stats:
            stats.add_log(
                event_time,
                event_type,
                operation,
                wait_value,
                execution_value,
                total_value,
                status
            )
    except (OSError, xmlrpc_client.Error, TimeoutError):
        pass

    return event


def wait_for_slot(operation):
    waiting_started = perf_counter()
    has_waited = False

    while True:
        now = monotonic()
        while REQUEST_TIMES and now - REQUEST_TIMES[0] >= WINDOW_SECONDS:
            REQUEST_TIMES.popleft()

        if len(REQUEST_TIMES) < MAX_REQUESTS:
            REQUEST_TIMES.append(now)
            return perf_counter() - waiting_started if has_waited else 0.0

        if not has_waited:
            add_event(
                'too_many_requests',
                operation,
                datetime.now().isoformat(timespec='microseconds'),
                status='rate_limited'
            )
            has_waited = True

        delay = max(0.001, REQUEST_TIMES[0] + WINDOW_SECONDS - monotonic())
        sleep(delay)


def make_proxy_method(method_name):
    def forward(*args):
        event_time = datetime.now().isoformat(timespec='microseconds')
        wait_seconds = wait_for_slot(method_name)
        execution_started = perf_counter()
        status = 'success'

        try:
            with xmlrpc_client.ServerProxy(MAIN_SERVER_URL, allow_none=True) as backend:
                result = getattr(backend, method_name)(*args)
            return result
        except Exception:
            status = 'error'
            raise
        finally:
            execution_seconds = perf_counter() - execution_started
            add_event(
                method_name,
                method_name,
                event_time,
                wait_seconds,
                execution_seconds,
                status
            )

    return forward


def get_proxy_events(limit=100):
    count = max(0, min(int(limit), EVENT_HISTORY_LIMIT))
    return list(EVENTS)[-count:] if count else []


def get_proxy_status():
    return {
        'proxy_port': PROXY_PORT,
        'main_server_url': MAIN_SERVER_URL,
        'stats2_url': STATS2_URL,
        'max_requests': MAX_REQUESTS,
        'window_seconds': WINDOW_SECONDS,
        'events_in_memory': len(EVENTS),
    }


## Запуск прокси-сервера

In [ ]:
proxy = SimpleXMLRPCServer(
    (PROXY_HOST, PROXY_PORT),
    requestHandler=RequestHandler,
    allow_none=True,
    logRequests=False
)

for name in METHODS:
    proxy.register_function(make_proxy_method(name), name)

proxy.register_function(get_proxy_events, 'get_proxy_events')
proxy.register_function(get_proxy_status, 'get_proxy_status')
proxy.register_introspection_functions()

print(f'Прокси: http://{PROXY_HOST}:{PROXY_PORT}/RPC2', flush=True)
print(f'Основной сервер: {MAIN_SERVER_URL}', flush=True)
print(f'Сервер статистики: {STATS2_URL}', flush=True)
print(f'Ограничение: {MAX_REQUESTS} запросов за {WINDOW_SECONDS:g} с', flush=True)
proxy.serve_forever()


Прокси: http://127.0.0.1:8009/RPC2
Основной сервер: http://127.0.0.1:8008/RPC2
Сервер статистики: http://127.0.0.1:8019/RPC2
Ограничение: 5 запросов за 5 с
